# Learned stable-note recovery: real data, measured results and experimental audio

**Current task:** test whether a bounded learned method can recover known local detuning on original human sustained vocals while preserving the original performance. We are narrowing the operating scope to a task with measurable context and enough support for the existing renderer. This is not an unrestricted correct/wrong singing classifier.

This notebook reads the new VocalSet source receipt and completed recovery evidence. It never downloads data, constructs interventions, fits a model, tunes thresholds or runs a broad search. Results appear only when the recovery run has written its completion pointer. If results are absent, the notebook reports that explicitly.

Earlier failed experiments remain in [06](06_real_voice_pilot_and_usefulness.ipynb); the original Transformer and constructed/noise demonstrations remain in [03](03_transformer_training.ipynb), [04](04_transformer_evaluation.ipynb) and [05](05_panel_demonstration.ipynb).

**Run:** select **BPDR local Python**, then **Restart Kernel and Run All Cells**. A known intervention establishes a recovery target; it does not establish the intended perfect melody of a natural performance.

In [ ]:
import os, sys, json, hashlib, zlib
from pathlib import Path
start=Path(os.environ.get('BPDR_COMPONENT_ROOT',Path.cwd())).resolve()
COMPONENT=next((p for p in (start,*start.parents) if p.name=='bpdr_input' and (p/'pyproject.toml').is_file()),None)
if COMPONENT is None: raise RuntimeError('Open inside BPDR_Workspace using the BPDR local Python kernel.')
WORKSPACE=COMPONENT/'BPDR_Workspace'
sys.path.insert(0,str(WORKSPACE/'source'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import display, Audio, Markdown

def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()
DATASET=WORKSPACE/'datasets/vocalset_pilot'
MANIFEST=read(DATASET/'manifest.json')
DATA_DECLARATION=read(DATASET/'declaration.json')
assert MANIFEST['status']=='COMPLETE'
assert sha(DATASET/'declaration.json')==MANIFEST['declaration_sha256']
print('Python:',sys.executable)
print('Workspace:',WORKSPACE)
print('Dataset manifest SHA-256:',sha(DATASET/'manifest.json'))
print('Notebook execution performs no model fitting or data download.')

## 1. Why the dataset and operating task changed

The previous small original-data pilot used short or fragmented reliable regions. In its Vocadito development subset, full injected waveform intervals were 0.74–0.96 seconds, with five of six below the unchanged controller's 0.86-second support requirement. The longest contiguous reliable scoring regions were only 0.16–0.18 seconds. Those scoring regions are distinct from full injected intervals.

The old endpoint also required raw original correction proposals to average at most one cent. Actual delivery additionally applies deadband, support selection, permission checks, rendering, remeasurement and rollback. A raw prediction failure and a delivered unwanted edit are different observations.

The earlier pilot remains failed under its original declaration. The new authorized demo experiment changes the source/task and declares its operating criteria before evaluation. It must report raw predictions **and** delivered results, including withheld cases and coverage. A visible local output difference alone cannot establish improvement.

VocalSet's long straight tones offer original human vocals with sustained pitch context. This narrower dataset does not represent natural novice errors, arbitrary speech, all expressive singing, or general melody correction.

The first VocalSet attempt also failed its declared calibration/coverage gates. Some known shifts landed in pauses or outside the method's own stable-voice scope. One further preparation correction is being declared to place interventions inside measured sustained voice while retaining clean context. The first attempt remains saved. Because the same evaluation singers were already inspected, the corrected attempt is a **development pilot, not an untouched final test**.

## 2. Official source, license and immutable originals

**Source:** [VocalSet: A Singing Voice Dataset, version 1.0](https://doi.org/10.5281/zenodo.1193957), Julia Wilkins, Prem Seetharaman, Alison Wahl and Bryan Pardo (2018).

**License:** [Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/). The local attribution is retained beside the source receipt. Any pitch-shifted derivative must be identified as modified and retain this attribution.

The bounded download selected the `long_tones/straight` recordings for vowels **a/e** from female1–6 and male1–6 before acoustic analysis. IDs 1–4 are training, ID 5 calibration, and ID 6 pilot evaluation, separately for the female/male groups. All 24 downloaded WAV originals are retained, including any later unusable files.

The complete 2 GB archive was not downloaded; its whole-archive MD5 was **not verified**. Instead, each extracted WAV is checked against its ZIP-entry CRC32 and pinned local SHA-256, and its decoded metadata is checked below. These are source-integrity checks, not acoustic-quality or musical-accuracy scores.

In [ ]:
source_rows=[]
for r in MANIFEST['records']:
    path=Path(r['path']).resolve()
    assert path.is_relative_to(DATASET.resolve())
    assert sha(path)==r['sha256'],path
    assert path.stat().st_size==r['bytes']
    assert format(zlib.crc32(path.read_bytes()) & 0xffffffff,'08x')==r['archive_crc32']
    assert r['archive_crc32_verified'] and not r['intervention_applied']
    info=sf.info(path)
    assert info.frames==r['sample_count'] and info.samplerate==r['sample_rate_hz'] and info.channels==r['channels']
    np.testing.assert_allclose(info.duration,r['duration_seconds'],rtol=0,atol=1e-9)
    source_rows.append(dict(split=r['split'],singer=r['singer_id'],vowel=r['vowel'],
        duration_seconds=r['duration_seconds'],sample_rate_hz=r['sample_rate_hz'],
        path=str(path),sha256=r['sha256']))
SOURCES=pd.DataFrame(source_rows)
assert len(SOURCES)==MANIFEST['recording_count']==24
assert SOURCES.singer.nunique()==MANIFEST['singer_count']==12
assert DATA_DECLARATION['license']=='CC-BY-4.0'
assert not DATA_DECLARATION['full_archive_md5_verified']
print((DATASET/'ATTRIBUTION.txt').read_text(encoding='utf-8'))
print('Verified all source WAV hashes, archive-entry CRCs and decoded metadata.')
display(SOURCES[['split','singer','vowel','duration_seconds','sample_rate_hz']])

## 3. Singer-disjoint training, calibration and pilot evaluation

Two vowels from one singer are dependent recordings. Later interventions and boundary variants remain dependent observations, not additional participants. Model fitting uses training singers; operating-point selection uses calibration singers only. Each run must state when model/threshold selection was frozen before that run's evaluation scoring. The evaluation singers were already inspected in the first failed attempt, so a later corrected run is developmental evidence.

Only two evaluation singers are available in this bounded pilot. Even favorable results cannot justify a population success rate or a stable generalization claim.

In [ ]:
splits=['train','calibration','pilot_evaluation']
groups={s:set(SOURCES[SOURCES.split==s].singer) for s in splits}
for i,a in enumerate(splits):
    for b in splits[i+1:]:
        assert not groups[a]&groups[b]
        assert not set(SOURCES[SOURCES.split==a].sha256)&set(SOURCES[SOURCES.split==b].sha256)
assert [len(groups[s]) for s in splits]==[8,2,2]
assert {s:int((SOURCES.split==s).sum()) for s in splits}==MANIFEST['split_counts']
summary=SOURCES.groupby('split').agg(original_recordings=('path','count'),singers=('singer','nunique'),
    audio_seconds=('duration_seconds','sum')).reindex(splits)
display(summary)
fig,ax=plt.subplots(figsize=(11,4))
colors={'train':'#477b88','calibration':'#b4884e','pilot_evaluation':'#9b645e'}
for split in splits:
    part=SOURCES[SOURCES.split==split]
    ax.scatter(part.singer,part.duration_seconds,label=split,color=colors[split],s=55,alpha=.8)
ax.set(ylabel='Original recording duration (seconds)',xlabel='Singer ID',title='Source coverage before acoustic eligibility or model scoring')
ax.legend();ax.tick_params(axis='x',rotation=25);plt.tight_layout();plt.show()
print('Disjoint singers:',{s:sorted(g) for s,g in groups.items()})

## 4. Hear two original training voices

These are unchanged official source WAVs from two training singers. Playing training sources avoids inspecting new evaluation acoustics before the recovery experiment finishes. The waveform plot is a source view, not a pitch-correctness assessment. Both players retain the recorded gain (`normalize=False`). No pitch extraction or model inference is needed for this source inspection.

In [ ]:
examples=[next(r for r in MANIFEST['records'] if r['singer_id']==s and r['vowel']=='a')
          for s in ['female1','male1']]
fig,axes=plt.subplots(2,1,figsize=(11,5),sharex=False)
for ax,r in zip(axes,examples):
    audio,sr=sf.read(r['path'],dtype='float32')
    take=np.arange(0,len(audio),max(1,sr//2000))
    ax.plot(take/sr,audio[take],linewidth=.5,color='#477b88')
    ax.set(ylabel='Amplitude',xlabel='Seconds',title=r['singer_id']+' / vowel '+r['vowel']+' / original recording')
    display(Markdown('**Original training voice: '+r['singer_id']+' / vowel '+r['vowel']+'**'))
    display(Audio(filename=r['path'],normalize=False))
plt.tight_layout();plt.show()

## 5. First frozen design and the retained failed attempt

Two fixed tree families and nine declared gain/deadband operating points are compared using training/calibration singers only. The selected model and operating point are locked before pilot-evaluation audio is prepared or scored. No calibration passer means the fallback remains diagnostic, even if evaluation happens to look favorable.

The first attempt used deterministic centre crops of at most eight seconds and resampling. Its separately declared preparation correction retains each complete source within the 20-second component limit and places shifts inside measured stable voice with context. Resampled and pitch-shifted working files are identified derivatives, while every official original is retained. Before/after recovery comparisons must use the same working sample grid. There are no natural intended-note labels.

The declared practical gate checks each evaluation singer separately, including positive delivered/permitted gain, response, raw and permitted original alteration, edit rate and nonzero coverage. Actual verified rendering and unchanged original anchors are additional requirements. Passing raw proposal metrics alone does not enable the candidate.

The cell below displays the **first** declaration and its failed outcome. The final completion record later in the notebook must identify any separately declared preparation correction. A new preparation rule does not erase failed cases or restore independent-test status.

In [ ]:
RECOVERY_DECLARATION_PATH=WORKSPACE/'results/model_recovery/14a10a583e40497ea2514fd3570a7f79/declaration.json'
if not RECOVERY_DECLARATION_PATH.is_file():
    raise FileNotFoundError('Restore the prospectively declared recovery protocol; this notebook does not create a replacement.')
RECOVERY_DECLARATION=read(RECOVERY_DECLARATION_PATH)
assert sha(DATASET/'manifest.json')==RECOVERY_DECLARATION['manifest_sha256']
print('Declaration time:',RECOVERY_DECLARATION['created_at'])
print('Declaration SHA-256:',sha(RECOVERY_DECLARATION_PATH))
print('Scope:',RECOVERY_DECLARATION['scope'])
print('Preprocessing:',RECOVERY_DECLARATION['preprocessing'])
print('Selection rule:',RECOVERY_DECLARATION['selection'])
print('Practical gates:',RECOVERY_DECLARATION['practical_gates'])
print('Compute caps:',RECOVERY_DECLARATION['caps'])
display(pd.DataFrame([{'model_family':name,**settings} for name,settings in RECOVERY_DECLARATION['models'].items()]))
display(pd.DataFrame(RECOVERY_DECLARATION['operating_points']))
print('Historical research gate preserved:',RECOVERY_DECLARATION['old_research_gate_preserved'])
FIRST_EVIDENCE=read(RECOVERY_DECLARATION_PATH.parent/'evidence.json')
first_outer=FIRST_EVIDENCE['outer']
display(pd.DataFrame([{'attempt':'First retained VocalSet run',
    'calibration_gate_passes':FIRST_EVIDENCE['selection']['calibration_gate_passes'],
    'outer_proposal_gate_passes':FIRST_EVIDENCE['outer_proposal_gate_passes'],
    'attempted_shifts':first_outer['attempted'],'audited_shifts':first_outer['audited'],
    'evaluation_singers_with_valid_labels':first_outer['groups'],
    'zero_mae_cents':first_outer.get('zero_mae'),
    'raw_correction_mae_cents':first_outer.get('raw_correction_mae'),
    'permitted_correction_mae_cents':first_outer.get('permitted_correction_mae'),
    'permitted_coverage':first_outer.get('permitted_detuned_nonidentity')}]))
assert not FIRST_EVIDENCE['outer_proposal_gate_passes']
print('First run retained as failed; no candidate adoption and no retrospective threshold edits.')

## 6. Read the completed recovery record

The completion pointer below is published by the separate bounded experiment, not by this notebook. Until it exists, there is no completed model result to report. The final walkthrough must verify source/model/result hashes and display all declared methods, selected calibration settings, held-out raw and delivered metrics, original preservation, coverage, permission withholding, rendering rollback and the unchanged baseline.

A rendered example must come from the frozen evaluation procedure and be traceable to its source and known intervention. The known original reference is for construction/scoring; it must not enter runtime proposal generation. A model type such as a tree ensemble must be named accurately and not called the Transformer or evidence of a BPDR advantage.

In [ ]:
RECOVERY_POINTER=WORKSPACE/'results/model_recovery/current.json'
RECOVERY=None
RECOVERY_DIRECTORY=None
if not RECOVERY_POINTER.is_file():
    print('Recovery experiment has not published a completed result. Source verification above is complete; no learned accuracy is claimed.')
else:
    pointer=read(RECOVERY_POINTER)
    candidate=pointer.get('path') or pointer.get('directory') or pointer.get('evidence_path')
    if not candidate: raise ValueError('Recovery completion pointer contains no result path.')
    location=Path(candidate)
    if not location.is_absolute(): location=WORKSPACE/location
    location=location.resolve()
    if not location.is_relative_to((WORKSPACE/'results/model_recovery').resolve()):
        raise ValueError('Recovery result leaves its declared workspace directory.')
    evidence_path=location if location.suffix=='.json' else location/'evidence.json'
    if not evidence_path.is_file(): raise FileNotFoundError('Completion pointer has no completed evidence.json')
    RECOVERY_DIRECTORY=evidence_path.parent
    RECOVERY=read(evidence_path)
    print('Recovery evidence:',evidence_path)
    print('Recovery evidence SHA-256:',sha(evidence_path))
    print('Evidence fields:',list(RECOVERY))
    print('Final metrics and rendered examples are populated from the completed schema below.')

## 7. Completed revised pilot: a trained model, with failed broader criteria

The corrected preparation retains full recordings and places controlled shifts in measured stable voice. Both fixed tree families trained successfully. A reporting-key error interrupted the first summary; the same deterministic fits were repeated from cached prepared data after a mechanical fix. The declaration, original executing-source snapshots and technical-resume receipt are retained.

**The broader calibration and two-singer criteria failed.** They remain failed. No research model was adopted. A separate, explicitly experimental demo option exposes a selected verified listening example; it does not promote this failed pilot into a successful validation study. The example was selected after observing the diagnostic outputs and is not representative accuracy.

The same two evaluation singer identities were reused after the preparation correction. They were absent from model fitting, but these results are **development evidence**, not an untouched final test. Only one supplied valid controlled-shift labels.

In [ ]:
assert RECOVERY is not None
FINAL_DECLARATION=read(RECOVERY_DIRECTORY/'declaration.json')
assert sha(RECOVERY_DIRECTORY/'declaration.json')==RECOVERY['declaration_sha256']
assert sha(DATASET/'manifest.json')==FINAL_DECLARATION['manifest_sha256']
CHECKPOINT=Path(RECOVERY['checkpoint'])
assert sha(CHECKPOINT)==RECOVERY['checkpoint_sha256']
from bpdr_input.modeling.recovery_model import load_locked_diagnostic
MODEL,PAYLOAD=load_locked_diagnostic(CHECKPOINT)
assert MODEL['kind']=='extra_trees'
assert not RECOVERY['outer_proposal_gate_passes']
assert not RECOVERY['rendered_demo_gate_passes']
assert not RECOVERY['model_adopted']
print('Model:',MODEL['kind'],'/ trees:',MODEL['estimator'].n_estimators)
print('Locked operating policy:',PAYLOAD['operating_policy'])
print('Model SHA-256:',sha(CHECKPOINT))
print('Full-recording preparation:',FINAL_DECLARATION['preprocessing'])
print('Technical resume:',read(RECOVERY_DIRECTORY/'technical_resume.json'))
display(pd.DataFrame(RECOVERY['fit_receipts']).T)
calibration=[]
for r in read(RECOVERY_DIRECTORY/'calibration_all.json'):
    s=r['summary']
    calibration.append(dict(model=r['model'],**r['operating_policy'],singers=s['groups'],
        raw_mae_cents=s.get('raw_correction_mae'),permitted_mae_cents=s.get('permitted_correction_mae'),
        raw_original_mean_cents=s.get('raw_original_abs_mean'),passes=s['passes']))
display(pd.DataFrame(calibration))
print('Selection was locked before this run prepared its outer examples; calibration gate passed:',RECOVERY['selection']['calibration_gate_passes'])
coverage=[]
for split in splits:
    pp=read(RECOVERY_DIRECTORY/(split+'_pairs.json'))
    ss=read(RECOVERY_DIRECTORY/(split+'_sources.json'))
    coverage.append(dict(split=split,originals=len(ss),signed_attempts=len(pp),
        audited=sum('excluded' not in p for p in pp),excluded=sum('excluded' in p for p in pp)))
display(pd.DataFrame(coverage))

## 8. Delivered WAV results: all attempts remain visible

An intervention is scoreable only after its independent construction audit passes. Twelve of sixteen evaluation interventions failed this prerequisite; their failures are retained and cannot supply target labels. All four scoreable interventions enter the delivered result, including withholding and an edit that did not improve the target.

The exact encoded WAV files were reloaded and audited. Sample count, timing, outside-support identity, pitch movement and clipping checks apply to the delivered bytes. These checks establish numerical behavior, not intended-note truth or a human quality judgment. The scored frame masks are fixed before output scoring; output cannot remove difficult frames from its denominator.

In [ ]:
VERIFICATION_PATH=RECOVERY_DIRECTORY/'rendered_verification.json'
assert sha(VERIFICATION_PATH)==RECOVERY['rendered_verification_sha256']
VERIFICATION=read(VERIFICATION_PATH)
assert VERIFICATION['checkpoint_sha256']==sha(CHECKPOINT)
assert VERIFICATION['declaration_sha256']==RECOVERY['declaration_sha256']
assert not VERIFICATION['rendered_demo_gate_passes']
assert len(VERIFICATION['originals'])==4 and len(VERIFICATION['shifts'])==16
for r in VERIFICATION['originals']:
    original,sr=sf.read(r['input'],dtype='float32')
    output,rate=sf.read(r['output'],dtype='float32')
    assert r['preserved'] and sr==rate and np.array_equal(original,output)
valid=[r for r in VERIFICATION['shifts'] if 'excluded' not in r]
assert len(valid)==4
for r in valid:
    report=read(Path(r['directory'])/'model_report.json')
    item=report['candidate'] if r['delivered_preview'] else report['handoff']['accepted_audio']
    assert sha(r['output'])==(item.get('sha256') or item['checksum'])
    if r['delivered_preview']:
        assert report['transaction']['delivered_wav_audit']['passed']
        assert report['transaction']['delivered_wav_audit']['sha256']==sha(r['output'])
    assert r['missing_output_scored_frames']==0
display(pd.DataFrame(VERIFICATION['shifts']).drop(columns=['input','reference','target_path','output','directory'],errors='ignore'))
display(pd.DataFrame(VERIFICATION['by_singer']).T)
print('Originals preserved:',sum(r['preserved'] for r in VERIFICATION['originals']),'of 4')
print('Verified improvements:',sum(r['verified_improvement'] for r in valid),'of 4 scoreable / 16 attempted interventions')
print('Declared broader gate:',VERIFICATION['rendered_demo_gate_passes'])
fig,ax=plt.subplots(figsize=(10,4))
x=np.arange(len(valid)); labels=[f"{r['singer']} / {Path(r['input']).parent.name} / {r['cents']:+g}c" for r in valid]
ax.bar(x-.18,[r['input_mae_cents'] for r in valid],.36,label='Input retained baseline',color='#9b645e')
ax.bar(x+.18,[r['output_mae_cents'] for r in valid],.36,label='Delivered output (withholds included)',color='#477b88')
ax.set(ylabel='Mean absolute local pitch error (cents)',title='All four scoreable development interventions; one singer')
ax.set_xticks(x,labels,rotation=15);ax.legend();plt.tight_layout();plt.show()

## 9. Hear the actual learned output

The listening example below is the **first verified improvement in the locked source/shift order**, selected after the diagnostic run. A +30-cent shift was added to a real recording; the learned ExtraTrees model receives only that shifted audio. It uses its own measured local pitch context, not the known clean reference, target mask, shift interval, C/D-scale label or requested correction.

The clean reference is used only to construct interventions and score output. The model does not know whether a natural small pitch movement was a mistake or intentional expression. User confirmation remains necessary before an experimental preview is exported. Audio players keep recorded gain.

In [ ]:
CARD_PATH=WORKSPACE/'results/model_recovery/demo_model.json'
CARD=read(CARD_PATH)
assert CARD['validation_status']=='exploratory_preview_failed_generalization_gate'
assert CARD['research_gates_passed'] is False and CARD['requires_user_review']
assert CARD['checkpoint_sha256']==sha(CHECKPOINT)
assert CARD['verification_sha256']==sha(VERIFICATION_PATH)
display(CARD['metrics'])
SELECTED=next(r for r in valid if r['verified_improvement'])
print('Selected example local MAE:',SELECTED['input_mae_cents'],'->',SELECTED['output_mae_cents'],'cents')
for label,key in [('Input: real voice with known added shift','input'),('Actual learned output WAV','output'),('Unshifted performance: scoring reference only','reference')]:
    display(Markdown('**'+label+'**'))
    display(Audio(filename=SELECTED[key],normalize=False))
from bpdr_input.repair.rule_correction import fine_pitch
with np.load(SELECTED['target_path']) as z:
    times=z['times']; fixed=z['mask'].astype(bool)&(z['target']!=0)
ref,sr=sf.read(SELECTED['reference'],dtype='float32')
before,_=sf.read(SELECTED['input'],dtype='float32')
after,_=sf.read(SELECTED['output'],dtype='float32')
reference_pitch=fine_pitch(ref,sr,times)
before_error=fine_pitch(before,sr,times)-reference_pitch
after_error=fine_pitch(after,sr,times)-reference_pitch
assert np.isfinite(before_error[fixed]).all() and np.isfinite(after_error[fixed]).all()
np.testing.assert_allclose(np.abs(before_error[fixed]).mean(),SELECTED['input_mae_cents'],atol=.001)
np.testing.assert_allclose(np.abs(after_error[fixed]).mean(),SELECTED['output_mae_cents'],atol=.001)
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(times[fixed],before_error[fixed],label='Input residual',color='#9b645e')
ax.plot(times[fixed],after_error[fixed],label='Output residual',color='#477b88')
ax.axhline(0,color='#333333',linewidth=.7)
ax.set(xlabel='Seconds from recording start (fixed scored frames)',ylabel='Cents relative to unshifted performance',title='Selected verified example; reference used after inference only')
ax.legend();plt.tight_layout();plt.show()

## 10. What can be said to the panel

- A small ExtraTrees model was trained on controlled modifications of original VocalSet human recordings, using singer-disjoint fitting and calibration groups. This model is not the historical Transformer and does not demonstrate a BPDR advantage.
- Two of four scoreable development interventions improved, both +30-cent shifts. Mean delivered local error across all four, including withholding and a non-improving edit, fell from 27.44 to 14.66 cents (46.6%). This is error reduction on four controlled examples from one singer, **not 46.6% general singing accuracy**.
- Four of four unshifted working recordings were preserved. Twelve of sixteen attempted interventions had no valid labels. Broader calibration and singer-coverage criteria failed; those failures remain unchanged.
- The demo exposes the model as an experimental optional preview. It does not automatically replace the accepted original. A separate user-reviewed package can proceed to the existing melody extractor when acoustic checks pass.
- The remaining work is more diverse reviewed singing, reliable training labels and independent singer validation. Arbitrary reading cannot establish an intended melody. Do not infer correctness solely from a presumed key or force every performance onto a scale.

**Demo:** run `Run-Demo.ps1`, select the experimental learned repair example, process, and compare input/output. The app runs inference from saved weights; it does not train while presenting.

In [ ]:
ui_path=WORKSPACE/'results/demo_sessions/ui_recovery_check.json'
if ui_path.is_file():
    UI=read(ui_path)
    assert UI['passed'] and UI['model_sha256']==sha(CHECKPOINT)
    assert UI['human_musical_judgment'] is False
    display(UI)
else:
    print('Live UI execution receipt is not yet available; no UI success is inferred.')
print('Notebook performed no fitting, download or intervention generation.')
print('Research model adoption remains:',RECOVERY['model_adopted'])
print('Current original manifest SHA-256:',sha(DATASET/'manifest.json'))